In [11]:
#finetuning summarization model
#basic imports
import pandas as pd
import numpy as np
import datasets
from datasets import load_dataset
import transformers

In [12]:
#dataset billsum
bill_sum = load_dataset('billsum',split='ca_test')

In [13]:
bill_sum

Dataset({
    features: ['text', 'summary', 'title'],
    num_rows: 1237
})

In [14]:
bill_sum = bill_sum.train_test_split(test_size=0.2)

In [15]:
bill_sum['train'][0]

{'text': 'The people of the State of California do enact as follows:\n\n\nSECTION 1.\nThe Legislature finds and declares all of the following:\n(a) The federal Patient Protection and Affordable Care Act provides millions of previously uninsured Californians access to health services, including physician care. As a result of this additional demand for physician services, the projected statewide physician shortfall is 17,000 for 2015.\n(b) The San Joaquin Valley, which runs from Stockton to Bakersfield, is rich in cultural diversity and is the nation’s leading agricultural region. However, the valley is disproportionately affected by the state’s physician shortage, which is expected to intensify in the years ahead given the high rate of population growth in the area. Access to health care is 31 percent lower in the San Joaquin Valley than in the rest of California.\n(c) Several regions of the San Joaquin Valley are federally designated Medically Underserved Areas (MUAs). The calculation 

### text and summary and title where summary is the label

### load tokenizer

In [16]:
from transformers import AutoTokenizer
check_point = 'google-t5/t5-small'
tokenizer = AutoTokenizer.from_pretrained(check_point)

In [17]:
prefix = "Summarise : "
def preprocess_function(examples):
    #join the prefix and the text
    inputs = [prefix + doc for doc in examples['text']]
    #create model input using tokenizer
    model_inputs = tokenizer(inputs,max_length=1024,truncation=True)
    #convert labels use text_target arg
    labels = tokenizer(text_target=examples['summary'],max_length=128,truncation=True)
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

In [18]:
#apply the function
tokenized_billsum = bill_sum.map(preprocess_function,batched=True)

Map:   0%|          | 0/989 [00:00<?, ? examples/s]

Map:   0%|          | 0/248 [00:00<?, ? examples/s]

### import data collator for seq2seq

In [19]:
from transformers import DataCollatorForSeq2Seq
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer,model=check_point)

In [20]:
!pip install evaluate rouge_score
import evaluate
rouge = evaluate.load('rouge')

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 10.3 MB/s eta 0:00:00
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=7a64d64f46da480945899e41c2a814d209555cf42f141473f3e7d39684cabfff
  Stored in directory: /root/.cache/pip/wheels/1e/19/43/8a442dc83660ca25e163e1bd1f89919284ab0d0c1475475148
Successfully built rouge_score
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.5.1
    Uninstalling fsspec-2025.5.1:
      Successfully uninstalled fsspec-2025.5.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.8.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which

In [21]:
#for compute metrics get the pred and the label decode using tokenizer.batch_decode and for 
# rouge score (prediction,refrence,use_stemmer)
def compute_metrics(eval_pred):
    #eval_pred is a tuple containing both pred and label
    predictions,labels = eval_pred
    #batch decode the prediction
    decoded_preds = tokenizer.batch_decode(predictions,skip_special_tokens=True)
    #labels conditionals
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    #decode
    decoded_labels = tokenizer.batch_decode(labels,skip_special_tokens=True)
    #calc rouge
    result = rouge.compute(predictions = decoded_preds,references = decoded_labels,use_stemmer = True)
    #lens
    gen_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in predictions ]
    result['gen_len'] = np.mean(gen_lens)
    #return results by rounding off
    return {k : round(v,4) for k,v in result.items()}

In [22]:
#laod dataset load tokenizer and model checkpoint, create preprocess function. which adds a prefix
# then combines text and summary both of which must be tokenized and combined into one 
#model input then map the function on the dataset get data collator
# then load rouge using evalute inside compute metrics get both predictions and labels
# use batch deocde and get decoded predictions and filter label from  padding and get decoded 
# create result and add mean generated length

### next steps load model, training_args and trainer itself

### for seq2seq task we specifically need particular models

In [23]:
from transformers import AutoModelForSeq2SeqLM, Seq2SeqTrainer, Seq2SeqTrainingArguments
#load model
model = AutoModelForSeq2SeqLM.from_pretrained(check_point)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [24]:
#training args
batch_size = 16
learning_rate = 2e-5
training_args = Seq2SeqTrainingArguments(
                output_dir = 'bill_sum/t5-fintuned',
                per_device_eval_batch_size= batch_size,
                per_device_train_batch_size= batch_size,
                learning_rate= learning_rate,
                weight_decay= 0.01,
                eval_strategy="epoch",
                num_train_epochs= 4,
                save_total_limit = 3,
                push_to_hub = False,
                predict_with_generate = True,
                report_to = [])

In [25]:
#trainer
trainer = Seq2SeqTrainer(model = model,
                        train_dataset= tokenized_billsum['train'],
                        eval_dataset = tokenized_billsum['test'],
                        processing_class = tokenizer,
                        compute_metrics = compute_metrics,
                        data_collator = data_collator,
                        args = training_args,
                        )

In [26]:
trainer.train()

Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum,Gen Len
1,No log,2.754175,0.118100,0.032100,0.099200,0.099300,20.000000
2,No log,2.554177,0.133300,0.046500,0.110100,0.110300,20.000000
3,No log,2.498995,0.137300,0.047400,0.112900,0.113000,20.000000
4,No log,2.483281,0.140900,0.050800,0.116100,0.116100,20.000000


TrainOutput(global_step=248, training_loss=3.062802222467238, metrics={'train_runtime': 239.9213, 'train_samples_per_second': 16.489, 'train_steps_per_second': 1.034, 'total_flos': 1070824333246464.0, 'train_loss': 3.062802222467238, 'epoch': 4.0})

In [29]:
trainer.push_to_hub()

Upload 3 LFS files:   0%|          | 0/3 [00:00<?, ?it/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

training_args.bin:   0%|          | 0.00/5.43k [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/Sharpaxis/t5-fintuned/commit/98827301826dd14288760eb38bcda603a929d258', commit_message='End of training', commit_description='', oid='98827301826dd14288760eb38bcda603a929d258', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Sharpaxis/t5-fintuned', endpoint='https://huggingface.co', repo_type='model', repo_id='Sharpaxis/t5-fintuned'), pr_revision=None, pr_num=None)

In [30]:
tokenizer.push_to_hub('Sharpaxis/t5-fintuned')

README.md: 0.00B [00:00, ?B/s]

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/Sharpaxis/t5-fintuned/commit/98827301826dd14288760eb38bcda603a929d258', commit_message='Upload tokenizer', commit_description='', oid='98827301826dd14288760eb38bcda603a929d258', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Sharpaxis/t5-fintuned', endpoint='https://huggingface.co', repo_type='model', repo_id='Sharpaxis/t5-fintuned'), pr_revision=None, pr_num=None)

In [28]:
from huggingface_hub import notebook_login
notebook_login()

In [34]:
#inference
from transformers import pipeline
summarizer = pipeline('summarization','Sharpaxis/t5-fintuned')

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


In [42]:
text = "summarize : " + bill_sum['train'][29]['text']

In [43]:
summarizer(text)

Token indices sequence length is longer than the specified maximum sequence length for this model (1321 > 512). Running this sequence through the model will result in indexing errors


[{'summary_text': 'the department of housing and community development report to the Legislature, no later than January 1, 2018, on ways to increase homeownership for extremely low, very low, and low-income households . the report to be submitted pursuant to subdivision (b) shall include, but are not limited to, the number of single-family homes owned by housing authorities in the last five years that were converted to ownership .'}]